# 39 · Stage-4 training runs (additional experiments) — session A (T4 GPU; run sessions A and B at the same time)
Up to 240 runs, primary schedule only, about 13 min each, in this order: E3 step 1 (block learning-rate factors 10 and 30, seed 0, 60 runs),
E4 wider encoding (60 runs), E1+ seeds 3–4 of the faster-learning models (60 runs), E3 step 2 (seeds 1–2 of the blocks whose selected
factor is 10 or 30; up to 60 runs; starts when all step-1 runs are finished). Cell 2 first runs a pre-flight check of the new models
(about 2 min) and stops if it fails.
Two copies of this notebook (A and B) share the work through lock files. Every run is trained in its own Python process
(`p2_run_stage3.py`, same code path as notebooks 13, 27 and 30), so GPU memory is released after each run (no out-of-memory build-up).
Each run is trained on the local disk of the runtime and copied to Drive once when it ends, so Drive Trash does not fill during training.
**Resumable:** after a disconnect simply **Run all** again; finished runs are skipped, an interrupted run restarts from epoch 0.
The prespecified text (cell 1) is stored with a UTC time stamp before the first run.

In [ ]:
# 1 · Settings, Drive, protocol record
SESSION = 'A'
from google.colab import drive
drive.mount('/content/drive', force_remount=True, timeout_ms=300000)
import os, sys, json, time, shutil, subprocess
PROJECT  = '/content/drive/MyDrive/quantum-placement'
CODE     = f'{PROJECT}/code'
CBIS_DIR = f'{PROJECT}/data/processed/cbis_mass_256_jit'
RES      = f'{PROJECT}/results'
OUT      = f'{RES}/p2_stage4'
ROOT     = f'{RES}/p2_stage4_runs_train'
ROOT3    = f'{RES}/p2_stage3_runs_train'
for d in (OUT, ROOT): os.makedirs(d, exist_ok=True)
PROTO = """Stage 4 - additional experiments (block learning rate chosen per block; five seeds for the faster-learning comparison; wider
encoding range), written before any stage-4 run was trained.

Motivation. In the faster-learning experiment (stage 3, E1) one untuned factor (100) set the effective learning rate of every 4->4
block, three seeds were used, and the codes entering the circuit covered a limited angle range (rotation-angle SD 0.56-0.68 rad).
Three additional experiments test whether these choices change the conclusions. The five-seed and stage-3 results stay as reported;
these experiments are reported as additional experiments, specified before their runs.

A. Training runs. Code, data (400 CBIS-DDSM mass crops, folds.csv), settings and evaluation as for the stage-3 runs: batch 16, Adam
   1e-4, up to 200 epochs, best checkpoint and early stopping (patience 10, from epoch 15) on the validation loss, learning rate halved
   after four epochs without improvement of the training loss (primary schedule 'train' only), Google Colab NVIDIA T4, one Python
   process per run (runner p2_run_stage4.py, models in qseg2_stage4.py). Run folders: results/p2_stage4_runs_train. Every model
   starts from the initial weights of the standard circuit model of the same placement and seed in every shared layer, and its
   effective initial block weights equal those of the standard block of the same kind and seed (checked before training).
   E3 - block learning rate chosen per block. Models g_/p_ x q_/cl_/tr_ x fast10/fast30: effective block learning rate 10 or 30
        times that of the rest of the network (reparameterization as in E1; factor 100 = the E1 models).
        Step 1 (selection, 60 runs): the 12 models, folds 0-4, seed 0. For each of the six blocks (placement x circuit, Dense(4),
        trigonometric) the factor among 10, 30 and 100 with the lowest mean best validation loss over the five folds at seed 0 is
        selected (factor 100: the E1 runs of seed 0). The selection uses validation losses only.
        Step 2 (at most 60 runs): for every block whose selected factor is 10 or 30, the same model with seeds 1 and 2, folds 0-4.
   E1+ - five seeds for the faster-learning comparison (60 runs): the six E1 models (g_/p_ x q_fast, cl_fast, tr_fast), folds 0-4,
        seeds 3 and 4, exactly as in E1.
   E4 - wider encoding range (60 runs): g_q_wide, p_q_wide, g_tr_wide, p_tr_wide; folds 0-4; seeds 0-2. The angle encoding of the
        circuit and of the trigonometric layer is pi*c instead of (pi/2)*c (codes c in (-1, 1), so angles span (-pi, pi)); block
        learning rate factor 100 as in E1; all else as E1. Dense(4) has no angle encoding, so cl_fast (E1) is its comparator.
   Run order: E3 step 1, E4, E1+, E3 step 2. A run that stops because of an infrastructure error is restarted from epoch 0; a run
   whose validation loss is never finite is recorded as failed and reported. All runs are reported; none is discarded or repeated
   for its result. No further stage-4 models, seeds, schedules, factors or encodings are added.

B. Test sets and evaluation: as stage 3 (CBIS-DDSM test folds, frozen CBIS-DDSM cohort, DMID, CSAW-S with the consensus and the three
   per-radiologist references, CDD-CESM), threshold 0.5, per-image Dice, IoU, HD95 and empty flag; on the sets other than the test
   folds, Dice is averaged per crop and seed over the five fold models.

C. Estimands and analysis (crossed hierarchical bootstrap over seeds, resampled jointly, and clusters - patients; DMID: source images -
   B = 10,000, numpy default_rng(0); verdicts as before: equivalent when the 95% CI lies within +-0.02 Dice, difference when it
   excludes zero, inconclusive otherwise; nominal when Holm-adjusted p >= 0.05).
   E3, per placement and test set, seeds 0-2, each block at its selected factor: q_sel - cl_sel and q_sel - tr_sel (Holm within
       these four contrasts of each test set). Descriptive: validation losses and test Dice of every block at every factor (seed 0).
   E1+, per placement and test set, seeds 0-4: q_fast - cl_fast and q_fast - tr_fast (Holm within these four contrasts of each test
       set); these five-seed values are reported with the three-seed E1 values, which are not replaced.
   E4, per placement and test set, seeds 0-2: q_wide - cl_fast and q_wide - tr_wide (Holm within these four contrasts of each test
       set). Descriptive: q_wide - q_fast and tr_wide - tr_fast; spread of the encoded angles; for the circuit models, the share of
       output variance explained by an additive model of the inputs (as in the five-seed study).
   Block weight change of every model is reported. If any primary contrast on the CBIS-DDSM test folds favors the circuit with a
   95% CI excluding zero, this is reported in the abstract; all other outcomes are reported in the results and the Online Resource."""
proto = f'{OUT}/PROTOCOL_stage4_prespecified.txt'
if not os.path.exists(proto):
    open(proto, 'w').write(PROTO + '\nwritten (UTC): ' + time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime()) + '\n')
assert open(proto).read().startswith(PROTO), 'protocol file on Drive differs from this notebook'
print(open(proto).read().splitlines()[-1])

In [ ]:
# 2 · GPU check (T4 only), runner script, local copy of the CBIS-DDSM crops
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
print('GPU:', gpu)
assert 'T4' in gpu, f'not a T4 ({gpu}): Runtime -> Change runtime type -> T4 GPU, then Run all'
for f in ('qseg2.py', 'qseg2_rev4.py'):
    assert os.path.exists(f'{CODE}/{f}'), f'{CODE}/{f} missing'
for f, src in (('qseg2_stage3.py', '"""Stage-3 additions to qseg2 (additional experiments, prespecified in protocols/PROTOCOL_stage3_additional_experiments.txt).\n\nE1 - faster-learning slot. The 4->4 block learns with an effective learning rate SLOT_SCALE (100) times that of the rest of the\n     network. This is done by reparameterization, so one Adam optimizer and the usual ReduceLROnPlateau schedule still apply:\n     every slot weight is stored as v = theta / k and used as theta = k * v. Adam moves v by about the learning rate per step,\n     so theta moves about k times as far. The initial theta is exactly that of the standard model of the same seed.\n     Models: g_q_fast, p_q_fast (circuit), g_cl_fast, p_cl_fast (Dense(4)), g_tr_fast, p_tr_fast (trigonometric).\nE2 - shared initialization. The trigonometric and identity controls start from the initial weights of the circuit model of the\n     same placement and seed in every layer they share with it (encoder, pre-slot, post-slot and decoder layers).\n     Models: g_tr_shared, p_tr_shared, g_id_shared, p_id_shared. The E1 trigonometric models are initialized the same way.\n\nLayer names are those of the standard models (quantum_circuit, classical_replacement, trig_replacement), so the standard\nevaluation code applies; the effective slot weights are SLOT_SCALE times the stored variables of the *_fast models.\nCall patch() once after importing qseg2 and qseg2_rev4 (qseg2_rev4.patch() is called here).\n"""\nimport numpy as np\nimport tensorflow as tf\nfrom tensorflow.keras import layers\n\nimport qseg2\nimport qseg2_rev4\n\nSLOT_SCALE = 100.0\nSLOT_LAYERS = (\'quantum_circuit\', \'classical_replacement\', \'trig_replacement\')\nFAST = (\'g_q_fast\', \'p_q_fast\', \'g_cl_fast\', \'p_cl_fast\', \'g_tr_fast\', \'p_tr_fast\')\nSHARED = (\'g_tr_shared\', \'p_tr_shared\', \'g_id_shared\', \'p_id_shared\')\nNEEDS_SHARED_INIT = (\'g_tr_fast\', \'p_tr_fast\') + SHARED\nAUDIT_LAYERS = SLOT_LAYERS + (\'pre_dense\', \'pre_conv\', \'post_dense\', \'post_conv\', \'bottleneck_conv2\', \'dec1_conv1\')\n\n\ndef _shrink(layer, names):\n    for n in names:\n        v = getattr(layer, n)\n        v.assign(v / SLOT_SCALE)\n\n\nclass FastCircuit(qseg2.TFCircuitLayer):\n    def build(self, input_shape):\n        super().build(input_shape)\n        _shrink(self, [\'qweights\'])\n\n    def call(self, inputs):\n        from qseg.quantum_sim import circuit_expvals, tf_ops\n        return circuit_expvals(tf.cast(inputs, tf.float32), SLOT_SCALE * tf.cast(self.qweights, tf.float32), tf_ops(), self.n_qubits)\n\n\nclass FastPatchCircuit(FastCircuit):\n    def call(self, inputs):\n        x = tf.cast(inputs, tf.float32)\n        shape = tf.shape(x)\n        out = super().call(tf.reshape(x, [-1, self.n_qubits]))\n        return tf.reshape(out, shape)\n\n    def compute_output_shape(self, input_shape):\n        return input_shape\n\n\nclass FastDense(layers.Dense):\n    def build(self, input_shape):\n        super().build(input_shape)\n        _shrink(self, [\'kernel\', \'bias\'])\n\n    def call(self, inputs):\n        y = tf.tensordot(tf.cast(inputs, tf.float32), SLOT_SCALE * self.kernel, axes=[[-1], [0]]) + SLOT_SCALE * self.bias\n        return tf.tanh(y)\n\n\nclass FastTrig(qseg2_rev4.TrigLayer):\n    def build(self, input_shape):\n        super().build(input_shape)\n        _shrink(self, [\'A\', \'phi\'])\n\n    def call(self, x):\n        c = tf.cos(np.pi / 2 * tf.cast(x, tf.float32) + SLOT_SCALE * self.phi)\n        return tf.tanh(tf.einsum(\'...i,ji->...j\', c, SLOT_SCALE * self.A))\n\n\n_prev_block = None\n\n\ndef _block(z, kind, n_qubits, n_layers, patch):\n    if kind == \'q_fast\':\n        return (FastPatchCircuit if patch else FastCircuit)(n_qubits, n_layers, name=\'quantum_circuit\')(z)\n    if kind == \'cl_fast\':\n        return FastDense(n_qubits, activation=\'tanh\', name=\'classical_replacement\')(z)\n    if kind == \'tr_fast\':\n        return FastTrig(n_qubits, name=\'trig_replacement\')(z)\n    if kind in (\'tr_shared\', \'id_shared\'):\n        return _prev_block(z, kind.split(\'_\')[0], n_qubits, n_layers, patch)\n    return _prev_block(z, kind, n_qubits, n_layers, patch)\n\n\n# ----------------------------------------------------------------------------- shared initialization\n_base_build = None\nINIT_SEED = None          # set by the runner before run_fold; consumed by the first model build\nINIT_RECORD = {}          # filled at that build: copy check and initial weights of the audited layers\n\n\ndef effective(layer, name):\n    """Effective weights of a layer as one flat float64 vector (slot of *_fast models: SLOT_SCALE x stored)."""\n    w = np.concatenate([v.numpy().astype(np.float64).ravel() for v in layer.trainable_weights]) if layer.trainable_weights else np.zeros(0)\n    if name in FAST and layer.name in SLOT_LAYERS:\n        w = w * SLOT_SCALE\n    return w\n\n\ndef build_p2_model(name, *args, **kwargs):\n    global INIT_SEED\n    model = _base_build(name, *args, **kwargs)\n    if INIT_SEED is None or name not in FAST + SHARED:\n        return model\n    seed, INIT_SEED = INIT_SEED, None\n    ref_name = name[:2] + \'q\'\n    qseg2.set_seeds(seed)\n    ref = _base_build(ref_name, *args, **kwargs)                  # standard circuit model of the same placement and seed\n    refw = {l.name: l.get_weights() for l in ref.layers if l.weights}\n    copied, before = [], {}\n    for l in model.layers:\n        if not l.weights or l.name not in refw:\n            continue\n        if l.name in SLOT_LAYERS:\n            if name.startswith((\'g_q\', \'p_q\')):                    # circuit: same initial angles as the standard model\n                w = refw[l.name]\n                l.set_weights([x / SLOT_SCALE for x in w] if name in FAST else w)\n            continue\n        a, b = l.get_weights(), refw[l.name]\n        if [x.shape for x in a] != [x.shape for x in b]:\n            continue\n        before[l.name] = float(max(np.abs(x - y).max() for x, y in zip(a, b)))\n        if name in NEEDS_SHARED_INIT:\n            l.set_weights(b); copied.append(l.name)\n    after = {l.name: float(max(np.abs(x - y).max() for x, y in zip(l.get_weights(), refw[l.name])))\n             for l in model.layers if l.weights and l.name in refw and l.name not in SLOT_LAYERS and l.name in before}\n    del ref\n    qseg2.set_seeds(seed)                                          # training randomness starts from the seed, as in all runs\n    INIT_RECORD.clear()\n    INIT_RECORD.update(reference=ref_name, seed=seed, copied_layers=copied, max_diff_before_copy=before,\n                       max_diff_after_copy=after, shared_identical=bool(all(v == 0 for v in after.values())),\n                       slot_scale=SLOT_SCALE if name in FAST else 1.0,\n                       init={l.name: effective(l, name) for l in model.layers if l.name in AUDIT_LAYERS})\n    return model\n\n\ndef weight_change(model, name):\n    """Change of the audited layers from INIT_RECORD[\'init\'] to the current (trained) weights."""\n    out = {}\n    for l in model.layers:\n        if l.name not in INIT_RECORD.get(\'init\', {}):\n            continue\n        a = INIT_RECORD[\'init\'][l.name]; b = effective(l, name); d = b - a\n        if l.name == \'quantum_circuit\':\n            d = (d + np.pi) % (2 * np.pi) - np.pi\n        out[l.name] = dict(n=int(a.size), max_abs_change=float(np.abs(d).max()), rms_change=float(np.sqrt((d ** 2).mean())),\n                           rel_change=float(np.linalg.norm(d) / max(np.linalg.norm(a), 1e-12)))\n    return out\n\n\ndef patch():\n    global _prev_block, _base_build\n    qseg2_rev4.patch()\n    if _prev_block is None:\n        _prev_block = qseg2._block\n        qseg2._block = _block\n    if _base_build is None:\n        _base_build = qseg2.build_p2_model\n        qseg2.build_p2_model = build_p2_model\n    lab = {\'q\': \'circuit\', \'cl\': \'Dense(4)\', \'tr\': \'trigonometric\', \'id\': \'identity\'}\n    for n in FAST:\n        qseg2.MODELS2.setdefault(n, f"{\'Global\' if n[0] == \'g\' else \'Patch-wise\'} bottleneck, {lab[n[2:].split(\'_\')[0]]}, slot learning rate x{SLOT_SCALE:g}")\n    for n in SHARED:\n        qseg2.MODELS2.setdefault(n, f"{\'Global\' if n[0] == \'g\' else \'Patch-wise\'} bottleneck, {lab[n[2:].split(\'_\')[0]]}, circuit-model initialization")\n    return qseg2\n'), ('qseg2_stage4.py', '"""Stage-4 additions to qseg2 (additional experiments, prespecified in protocols/PROTOCOL_stage4_additional_experiments.txt).\n\nE3 - block learning rate chosen per block. As E1 (qseg2_stage3), but with effective block learning-rate factors 10 and 30:\n     models g_/p_ x q_/cl_/tr_ x fast10/fast30 (e.g. g_cl_fast10). The factor-100 models are the E1 models (*_fast).\nE4 - wider encoding range. Circuit and trigonometric blocks whose angle encoding is pi*c instead of (pi/2)*c, with the\n     factor-100 block learning rate of E1: g_q_wide, p_q_wide, g_tr_wide, p_tr_wide. The Dense(4) block has no encoding,\n     so g_cl_fast / p_cl_fast (E1) are its comparators.\nEvery model starts from the initial weights of the standard circuit model of the same placement and seed in every layer it\nshares with it (as the E1 models; checked before training). Layer names are those of the standard models.\nCall patch() once after importing qseg2 (qseg2_stage3.patch() is called here).\n"""\nimport numpy as np\nimport tensorflow as tf\nfrom tensorflow.keras import layers\n\nimport qseg2\nimport qseg2_rev4\nimport qseg2_stage3 as S3\n\nENC_WIDE = 2.0                                   # angle = (pi/2) * ENC * c, so ENC = 2 gives pi * c\nFACTORS = {\'fast10\': 10.0, \'fast30\': 30.0, \'wide\': 100.0}\nNEW = tuple(f\'{p}_{b}_{v}\' for p in \'gp\' for b in (\'q\', \'cl\', \'tr\') for v in (\'fast10\', \'fast30\')) + \\\n      tuple(f\'{p}_{b}_wide\' for p in \'gp\' for b in (\'q\', \'tr\'))\n\n\ndef scale_of(name):\n    """Effective slot learning-rate factor of a model (1 for standard models)."""\n    v = name.split(\'_\', 2)[-1] if name.count(\'_\') >= 2 else \'\'\n    if v in FACTORS: return FACTORS[v]\n    if v == \'fast\': return S3.SLOT_SCALE\n    return 1.0\n\n\ndef enc_of(name):\n    return ENC_WIDE if name.endswith(\'_wide\') else 1.0\n\n\ndef _shrink(layer, names, k):\n    for n in names:\n        v = getattr(layer, n); v.assign(v / k)\n\n\nclass ScaledCircuit(qseg2.TFCircuitLayer):\n    def __init__(self, *a, k=1.0, enc=1.0, **kw):\n        super().__init__(*a, **kw); self.k, self.enc = float(k), float(enc)\n\n    def build(self, input_shape):\n        super().build(input_shape); _shrink(self, [\'qweights\'], self.k)\n\n    def call(self, inputs):\n        from qseg.quantum_sim import circuit_expvals, tf_ops\n        return circuit_expvals(self.enc * tf.cast(inputs, tf.float32), self.k * tf.cast(self.qweights, tf.float32), tf_ops(), self.n_qubits)\n\n\nclass ScaledPatchCircuit(ScaledCircuit):\n    def call(self, inputs):\n        x = tf.cast(inputs, tf.float32); shape = tf.shape(x)\n        return tf.reshape(super().call(tf.reshape(x, [-1, self.n_qubits])), shape)\n\n    def compute_output_shape(self, input_shape):\n        return input_shape\n\n\nclass ScaledDense(layers.Dense):\n    def __init__(self, *a, k=1.0, **kw):\n        super().__init__(*a, **kw); self.k = float(k)\n\n    def build(self, input_shape):\n        super().build(input_shape); _shrink(self, [\'kernel\', \'bias\'], self.k)\n\n    def call(self, inputs):\n        return tf.tanh(tf.tensordot(tf.cast(inputs, tf.float32), self.k * self.kernel, axes=[[-1], [0]]) + self.k * self.bias)\n\n\nclass ScaledTrig(qseg2_rev4.TrigLayer):\n    def __init__(self, *a, k=1.0, enc=1.0, **kw):\n        super().__init__(*a, **kw); self.k, self.enc = float(k), float(enc)\n\n    def build(self, input_shape):\n        super().build(input_shape); _shrink(self, [\'A\', \'phi\'], self.k)\n\n    def call(self, x):\n        c = tf.cos(np.pi / 2 * self.enc * tf.cast(x, tf.float32) + self.k * self.phi)\n        return tf.tanh(tf.einsum(\'...i,ji->...j\', c, self.k * self.A))\n\n\n_prev_block = None\n\n\ndef _block(z, kind, n_qubits, n_layers, patch):\n    base, _, var = kind.partition(\'_\')\n    if var in FACTORS:\n        k, enc = FACTORS[var], (ENC_WIDE if var == \'wide\' else 1.0)\n        if base == \'q\':\n            return (ScaledPatchCircuit if patch else ScaledCircuit)(n_qubits, n_layers, k=k, enc=enc, name=\'quantum_circuit\')(z)\n        if base == \'cl\':\n            return ScaledDense(n_qubits, activation=\'tanh\', k=k, name=\'classical_replacement\')(z)\n        if base == \'tr\':\n            return ScaledTrig(n_qubits, k=k, enc=enc, name=\'trig_replacement\')(z)\n    return _prev_block(z, kind, n_qubits, n_layers, patch)\n\n\ndef effective(layer, name):\n    w = np.concatenate([v.numpy().astype(np.float64).ravel() for v in layer.trainable_weights]) if layer.trainable_weights else np.zeros(0)\n    if layer.name in S3.SLOT_LAYERS:\n        w = w * scale_of(name)\n    return w\n\n\n_s3_build = None\n\n\ndef build_p2_model(name, *args, **kwargs):\n    """New models: built, then every layer shared with the standard circuit model of the same placement and seed is copied\n    from it, and the block gets the standard initial values (scaled). Other models: unchanged (stage-3 code path)."""\n    if name not in NEW or S3.INIT_SEED is None:\n        return _s3_build(name, *args, **kwargs)\n    seed, S3.INIT_SEED = S3.INIT_SEED, None\n    model = S3._base_build(name, *args, **kwargs)\n    k = scale_of(name)\n    qseg2.set_seeds(seed)\n    ref = S3._base_build(name[:2] + \'q\', *args, **kwargs)                      # standard circuit model, same seed\n    refw = {l.name: l.get_weights() for l in ref.layers if l.weights}\n    if not name.startswith((\'g_q\', \'p_q\')):                                      # standard block of this kind, same seed\n        qseg2.set_seeds(seed)\n        own = S3._base_build(name[:2] + name[2:].split(\'_\')[0], *args, **kwargs)\n        ownw = {l.name: l.get_weights() for l in own.layers if l.name in S3.SLOT_LAYERS}\n        del own\n    else:\n        ownw = refw\n    copied, before = [], {}\n    for l in model.layers:\n        if not l.weights:\n            continue\n        if l.name in S3.SLOT_LAYERS:\n            l.set_weights([x / k for x in ownw[l.name]]); continue\n        if l.name not in refw:\n            continue\n        a, b = l.get_weights(), refw[l.name]\n        if [x.shape for x in a] != [x.shape for x in b]:\n            continue\n        before[l.name] = float(max(np.abs(x - y).max() for x, y in zip(a, b)))\n        l.set_weights(b); copied.append(l.name)\n    after = {l.name: float(max(np.abs(x - y).max() for x, y in zip(l.get_weights(), refw[l.name])))\n             for l in model.layers if l.name in before}\n    del ref\n    qseg2.set_seeds(seed)\n    S3.INIT_RECORD.clear()\n    S3.INIT_RECORD.update(reference=name[:2] + \'q\', seed=seed, copied_layers=copied, max_diff_before_copy=before,\n                          max_diff_after_copy=after, shared_identical=bool(all(v == 0 for v in after.values())),\n                          slot_scale=k, encoding_factor=enc_of(name),\n                          init={l.name: effective(l, name) for l in model.layers if l.name in S3.AUDIT_LAYERS})\n    return model\n\n\ndef weight_change(model, name):\n    out = {}\n    for l in model.layers:\n        if l.name not in S3.INIT_RECORD.get(\'init\', {}):\n            continue\n        a = S3.INIT_RECORD[\'init\'][l.name]; b = effective(l, name); d = b - a\n        if l.name == \'quantum_circuit\':\n            d = (d + np.pi) % (2 * np.pi) - np.pi\n        out[l.name] = dict(n=int(a.size), max_abs_change=float(np.abs(d).max()), rms_change=float(np.sqrt((d ** 2).mean())),\n                           rel_change=float(np.linalg.norm(d) / max(np.linalg.norm(a), 1e-12)))\n    return out\n\n\ndef patch():\n    global _prev_block, _s3_build\n    S3.patch()\n    if _prev_block is None:\n        _prev_block = qseg2._block; qseg2._block = _block\n    if _s3_build is None:\n        _s3_build = qseg2.build_p2_model; qseg2.build_p2_model = build_p2_model\n    lab = {\'q\': \'circuit\', \'cl\': \'Dense(4)\', \'tr\': \'trigonometric\'}\n    for n in NEW:\n        b, v = n[2:].split(\'_\')\n        desc = f\'slot learning rate x{FACTORS[v]:g}\' + (\', encoding pi*c\' if v == \'wide\' else \'\')\n        qseg2.MODELS2.setdefault(n, f"{\'Global\' if n[0] == \'g\' else \'Patch-wise\'} bottleneck, {lab[b]}, {desc}")\n    return qseg2\n'), ('check_stage4b.py', '"""Pre-flight check of the stage-4 models (no data, about a minute on a GPU). Stops with an error if any check fails.\n\nFor every new model (seed 1): (1) the effective initial block weights equal those of the standard block of the same kind and\nseed; (2) for the circuit and Dense(4) learning-rate variants, the output equals that of the standard model on random input\n(the standard trigonometric model has its own decoder initialization, so its output differs by design); (3) every layer shared\nwith the standard circuit model is identical; (4) one Adam step (learning rate 1e-4) moves the effective block weights about\nk times as far as in the standard model (k = 10, 30 or 100).\n"""\nimport json\nimport numpy as np\nimport tensorflow as tf\nimport qseg2\nimport qseg2_stage3 as S3\nimport qseg2_stage4 as S4\n\nS4.patch()\nSHAPE = (256, 256, 1)\nrng = np.random.default_rng(0)\nx = rng.random((2,) + SHAPE).astype(\'float32\'); y = (rng.random((2,) + SHAPE) > 0.7).astype(\'float32\')\nreport = {}\n\n\ndef build(name, seed, copy):\n    tf.keras.backend.clear_session()\n    qseg2.set_seeds(seed)\n    S3.INIT_SEED = seed if copy else None\n    return qseg2.build_p2_model(name, SHAPE, 4, 2)\n\n\ndef slot(m, name):\n    for l in m.layers:\n        if l.name in S3.SLOT_LAYERS:\n            return S4.effective(l, name)\n\n\ndef one_step(m):\n    opt = tf.keras.optimizers.Adam(1e-4)\n    with tf.GradientTape() as t:\n        loss = qseg2.bce_dice_loss(y, m(x, training=False))\n    g = t.gradient(loss, m.trainable_weights)\n    opt.apply_gradients(zip(g, m.trainable_weights))\n\n\nfor name in S4.NEW:\n    std = name[:2] + name[2:].split(\'_\')[0]\n    k = S4.scale_of(name)\n    ms = build(std, 1, False); a0 = slot(ms, std); out_s = ms(x, training=False).numpy(); one_step(ms); da = np.abs(slot(ms, std) - a0).max()\n    mf = build(name, 1, True); rec = dict(S3.INIT_RECORD)\n    b0 = slot(mf, name); out_f = mf(x, training=False).numpy(); one_step(mf); db = np.abs(slot(mf, name) - b0).max()\n    r = dict(k=k, init_max_diff=float(np.abs(a0 - b0).max()), output_max_diff=float(np.abs(out_s - out_f).max()),\n             ratio=float(db / max(da, 1e-12)), copied=len(rec.get(\'copied_layers\', [])), shared_identical=rec.get(\'shared_identical\'))\n    report[name] = r\n    assert r[\'init_max_diff\'] < 1e-5, (name, r)\n    if not name.endswith(\'_wide\') and name[2:].split(\'_\')[0] in (\'q\', \'cl\'):\n        assert r[\'output_max_diff\'] < 1e-4, (name, r)\n    assert k / 3 < r[\'ratio\'] < 3 * k, (name, r)\n    assert r[\'shared_identical\'] and r[\'copied\'] > 10, (name, r)\n\nprint(json.dumps(report, indent=1))\nprint(\'STAGE-4 CHECK PASSED\')\n')):
    if os.path.exists(f'{CODE}/{f}'): assert open(f'{CODE}/{f}').read() == src, f'{CODE}/{f} differs from this notebook'
    else: open(f'{CODE}/{f}', 'w').write(src)
# qseg package: located exactly as in notebooks 13 and 27 (project code folder first, otherwise the first qseg/quantum_sim.py on Drive)
import glob, hashlib
QSEG_DIR = CODE
if not os.path.exists(f'{QSEG_DIR}/qseg/quantum_sim.py'):
    hits = glob.glob('/content/drive/MyDrive/**/qseg/quantum_sim.py', recursive=True); assert hits, 'qseg package not found on Drive'
    QSEG_DIR = os.path.dirname(os.path.dirname(hits[0]))
QSEG_MD5 = {os.path.basename(f): hashlib.md5(open(f, 'rb').read()).hexdigest() for f in sorted(glob.glob(f'{QSEG_DIR}/qseg/*.py'))}
json.dump(dict(qseg_dir=QSEG_DIR, md5=QSEG_MD5), open(f'{OUT}/qseg_used_{SESSION}.json', 'w'), indent=1)
print('qseg package:', QSEG_DIR, '| files:', len(QSEG_MD5))
ENV = dict(os.environ, PYTHONPATH=os.pathsep.join([QSEG_DIR, CODE] + ([os.environ['PYTHONPATH']] if os.environ.get('PYTHONPATH') else [])))
RUNNER = f'{CODE}/p2_run_stage4.py'
open(RUNNER, 'w').write('"""One training run of the stage-4 additional experiments (models of src/qseg2_stage4.py; same code path as src/p2_run_stage3.py).\nAfter training it writes init_record.json (shared-initialization check, slot scale) and weight_change.json\n(change of the slot and its neighbouring layers from initialization to the saved best weights).\n\nOriginal runner description:\n\n    python p2_run.py --model g_q --fold 0 --seed 3 --schedule train \\\n        --cbis /path/data/processed/cbis_mass_256_jit --out /path/results\n\nRuns go to <out>/p2_jit_runs_<schedule>/<model>_fold<k>_seed<s>/ (or to --root) exactly as in the notebooks (summary.json, history.csv,\nper-image CSV, best.weights.h5 without optimizer state). A finished run (summary.json present) is skipped, so a job array can\nsimply be resubmitted. Settings: Config defaults of the study with max_epochs=200; learning rate halved after 4 epochs without\nimprovement of the training loss (schedule \'train\') or of the validation loss (schedule \'val\'). The software and GPU of every\nrun are written to <run>/env.json.\n"""\nimport argparse, json, os, platform, subprocess, sys, time\n\nHERE = os.path.dirname(os.path.abspath(__file__))\nsys.path.insert(0, HERE)\n\nimport numpy as np\nimport tensorflow as tf\nimport qseg2\nimport qseg2_rev4\nimport qseg2_stage3\nimport qseg2_stage4\nfrom qseg.config import Config\n\nqseg2_stage4.patch()                       # adds g_tr / p_tr, the stage-3 and the stage-4 models\n\n\ndef use_schedule(schedule):\n    """As notebook 27, cell 1: ReduceLROnPlateau watches \'loss\' (train) or \'val_loss\' (val)."""\n    base = tf.keras.callbacks.ReduceLROnPlateau\n    while getattr(base, \'_p2_switch\', False):\n        base = base.__mro__[1]\n    monitor = {\'train\': \'loss\', \'val\': \'val_loss\'}[schedule]\n\n    class SwitchPlateau(base):\n        _p2_switch = True\n\n        def __init__(self, monitor_=None, **kw):\n            kw.pop(\'monitor\', None)\n            super().__init__(monitor=monitor, **kw)\n\n    tf.keras.callbacks.ReduceLROnPlateau = SwitchPlateau\n\n\ndef shrink_weights(cfg, run_dir, name):\n    """As notebook 27, cell 2: weight file without optimizer state, verified weight by weight."""\n    big, small = f\'{run_dir}/best.weights.h5\', f\'{run_dir}/best_small.weights.h5\'\n    if not os.path.exists(big) or os.path.exists(f\'{run_dir}/.shrunk\'):\n        return\n    if os.path.getsize(big) < 60e6:\n        open(f\'{run_dir}/.shrunk\', \'w\').close(); return\n    shape = (cfg.image_size, cfg.image_size, cfg.in_channels)\n    tf.keras.backend.clear_session()\n    m1 = qseg2.build_p2_model(name, shape, cfg.n_qubits, cfg.n_q_layers); m1.load_weights(big); m1.save_weights(small)\n    m2 = qseg2.build_p2_model(name, shape, cfg.n_qubits, cfg.n_q_layers); m2.load_weights(small)\n    assert all(np.array_equal(a, b) for a, b in zip(m1.get_weights(), m2.get_weights())), f\'{run_dir}: weight check failed\'\n    os.replace(small, big); open(f\'{run_dir}/.shrunk\', \'w\').close()\n\n\ndef env_info():\n    try:\n        gpu = subprocess.run([\'nvidia-smi\', \'--query-gpu=name,driver_version\', \'--format=csv,noheader\'],\n                             capture_output=True, text=True, timeout=30).stdout.strip()\n    except Exception:\n        gpu = \'unknown\'\n    b = tf.sysconfig.get_build_info()\n    return dict(python=platform.python_version(), tensorflow=tf.__version__, keras=tf.keras.__version__ if hasattr(tf.keras, \'__version__\') else \'\',\n                cuda=b.get(\'cuda_version\'), cudnn=b.get(\'cudnn_version\'), gpu=gpu, host=platform.node())\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\'--model\', required=True, choices=sorted(qseg2.MODELS2))\n    ap.add_argument(\'--fold\', type=int, required=True, choices=range(5))\n    ap.add_argument(\'--seed\', type=int, required=True)\n    ap.add_argument(\'--schedule\', default=\'train\', choices=(\'train\', \'val\'))\n    ap.add_argument(\'--cbis\', required=True, help=\'folder with folds.csv, images/, masks/\')\n    ap.add_argument(\'--out\', required=True, help=\'results root\')\n    ap.add_argument(\'--root\', default=None, help=\'run folder root (default <out>/p2_jit_runs_<schedule>)\')\n    a = ap.parse_args()\n\n    root = a.root or os.path.join(a.out, f\'p2_jit_runs_{a.schedule}\')\n    run = f\'{a.model}_fold{a.fold}_seed{a.seed}\'\n    d = os.path.join(root, run)\n    if os.path.exists(os.path.join(d, \'summary.json\')):\n        print(\'already finished:\', run); return\n    os.makedirs(d, exist_ok=True)\n    json.dump(env_info(), open(os.path.join(d, \'env.json\'), \'w\'), indent=2)\n\n    use_schedule(a.schedule)\n    cfg = Config(work_dir=a.out, max_epochs=200)\n    if not os.path.exists(os.path.join(root, \'train_config.json\')):    # never overwrite the record of earlier runs\n        cfg.to_json(os.path.join(root, \'train_config.json\'))\n    if not os.path.exists(os.path.join(root, \'schedule.json\')):\n        json.dump({\'schedule\': a.schedule}, open(os.path.join(root, \'schedule.json\'), \'w\'))\n    data = qseg2.load_cbis(a.cbis)\n    t0 = time.time()\n    if a.model in qseg2_stage4.NEW or a.model in qseg2_stage3.NEEDS_SHARED_INIT:   # check the shared initialization before training\n        qseg2_stage3.INIT_SEED = a.seed\n        qseg2.set_seeds(a.seed)\n        qseg2.build_p2_model(a.model, (cfg.image_size, cfg.image_size, cfg.in_channels), cfg.n_qubits, cfg.n_q_layers)\n        if not qseg2_stage3.INIT_RECORD.get(\'shared_identical\') or not qseg2_stage3.INIT_RECORD.get(\'copied_layers\'):\n            raise RuntimeError(\'shared initialization check failed\')\n        tf.keras.backend.clear_session()\n    qseg2_stage3.INIT_SEED = a.seed\n    s = qseg2.run_fold(cfg, data, a.model, a.fold, a.seed, out_root=root, min_epochs=15, verbose=2)\n    rec = {k: v for k, v in qseg2_stage3.INIT_RECORD.items() if k != \'init\'}\n    rec[\'slot_init_effective\'] = {k: v.tolist() for k, v in qseg2_stage3.INIT_RECORD.get(\'init\', {}).items() if k in qseg2_stage3.SLOT_LAYERS}\n    json.dump(rec, open(os.path.join(d, \'init_record.json\'), \'w\'), indent=1)\n    shrink_weights(cfg, d, a.model)\n    tf.keras.backend.clear_session()\n    m = qseg2.build_p2_model(a.model, (cfg.image_size, cfg.image_size, cfg.in_channels), cfg.n_qubits, cfg.n_q_layers)\n    m.load_weights(os.path.join(d, \'best.weights.h5\'))\n    json.dump(qseg2_stage4.weight_change(m, a.model), open(os.path.join(d, \'weight_change.json\'), \'w\'), indent=1)\n    print(f\'{run} [{a.schedule}]: Dice {s["dice_per_image"]:.4f}, {s["epochs_run"]} epochs, {(time.time() - t0) / 60:.1f} min\', flush=True)\n\n\nif __name__ == \'__main__\':\n    main()\n')
chk = f'{OUT}/preflight_check_{SESSION}.txt'
if not os.path.exists(chk) or 'PASSED' not in open(chk).read():
    res = subprocess.run([sys.executable, f'{CODE}/check_stage4b.py'], capture_output=True, text=True, env=ENV)
    open(chk, 'w').write(res.stdout + res.stderr[-5000:])
    print(res.stdout[-3000:], res.stderr[-3000:])
    assert 'STAGE-4 CHECK PASSED' in res.stdout, 'pre-flight check failed: send the output above'
print('pre-flight check passed')
LOCAL = '/content/cbis_mass_256_jit'
if not os.path.exists(f'{LOCAL}/folds.csv'): shutil.copytree(CBIS_DIR, LOCAL)
print('crops:', len(os.listdir(f'{LOCAL}/images')), '| runner written')

In [ ]:
# 3 · Training (sessions share the work through lock files; one process per run)
BLOCKS = [f'{p}_{b}' for p in 'gp' for b in ('q', 'cl', 'tr')]
STEP1 = [(f'{b}_{v}', f, 0, 'train') for b in BLOCKS for v in ('fast10', 'fast30') for f in range(5)]
E4 = [(m, f, s, 'train') for s in (0, 1, 2) for f in range(5) for m in ('g_q_wide', 'p_q_wide', 'g_tr_wide', 'p_tr_wide')]
E1P = [(f'{b}_fast', f, s, 'train') for s in (3, 4) for f in range(5) for b in BLOCKS]
RUNS = STEP1 + E4 + E1P
assert len(RUNS) == 180 and len(set(RUNS)) == 180
def root_of(m, sch): return ROOT
def rdir(m, f, s, sch): return f'{root_of(m, sch)}/{m}_fold{f}_seed{s}'
def finished(r): return os.path.exists(rdir(*r) + '/summary.json') or os.path.exists(rdir(*r) + '/failed.json')
LOCK_H = 2
def claim(r):
    d = rdir(*r)
    if finished(r): return False
    lk = f'{d}/.lock'
    if os.path.exists(lk):
        try: who, t = open(lk).read().split(); t = float(t)
        except Exception: who, t = SESSION, 0.0
        if who != SESSION and time.time() - t < LOCK_H * 3600: return False
    os.makedirs(d, exist_ok=True); open(lk, 'w').write(f'{SESSION} {time.time()}'); time.sleep(30)
    try: return open(lk).read().split()[0] == SESSION
    except Exception: return False
WORK = '/content/p2_work'     # each run is trained on the local disk and copied to Drive once at the end:
                              # no checkpoint or history file is overwritten on Drive during training (keeps Drive Trash small)
def train(r):
    m, f, s, sch = r; log = f'/content/log_{m}_{f}_{s}_{sch}.txt'
    lroot = f'{WORK}/{os.path.basename(root_of(m, sch))}'; ldir = f'{lroot}/{m}_fold{f}_seed{s}'
    shutil.rmtree(ldir, ignore_errors=True); os.makedirs(lroot, exist_ok=True)
    cmd = [sys.executable, RUNNER, '--model', m, '--fold', str(f), '--seed', str(s), '--schedule', sch,
           '--cbis', LOCAL, '--out', RES, '--root', lroot]
    with open(log, 'w') as fh: rc = subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT, env=ENV).returncode
    d = rdir(*r); os.makedirs(d, exist_ok=True)
    if os.path.isdir(ldir):
        names = sorted(os.listdir(ldir), key=lambda n: n == 'summary.json')      # summary.json last = run complete on Drive
        for n in names:
            src = f'{ldir}/{n}'
            if os.path.isdir(src): shutil.copytree(src, f'{d}/{n}', dirs_exist_ok=True)
            else: shutil.copy2(src, f'{d}/{n}')
        shutil.rmtree(ldir, ignore_errors=True)
    shutil.copy(log, d + f'/train_log_attempt{int(time.time())}.txt')
    return rc, open(log).read()[-3000:]
def select():
    """E3 step 1: per block, the factor (10, 30, 100) with the lowest mean best validation loss over folds 0-4 at seed 0."""
    sel = {}
    for b in BLOCKS:
        loss = {}
        for v, k in (('fast10', 10), ('fast30', 30), ('fast', 100)):
            root = ROOT if v != 'fast' else ROOT3
            vals = [json.load(open(f'{root}/{b}_{v}_fold{f}_seed0/summary.json'))['best_val_loss'] for f in range(5)]
            loss[k] = sum(vals) / 5
        sel[b] = dict(mean_best_val_loss=loss, selected=min(loss, key=loss.get))
    return sel
def run_list(runs):
    order = runs if SESSION == 'A' else runs[::-1]
    for r in order:
        yield r
STEP2 = None
def phase(runs):
    for r in (runs if SESSION == 'A' else runs[::-1]):
        if not claim(r): continue
        m, f, s, sch = r; t0 = time.time()
        rc, tail = train(r)
        if rc != 0 and not os.path.exists(rdir(*r) + '/summary.json'):
            if 'validation loss never finite' in tail:
                json.dump(dict(model=m, fold=f, seed=s, schedule=sch, failed='validation loss never finite'), open(rdir(*r) + '/failed.json', 'w'))
            else:
                print(f'   {m} f{f} s{s} [{sch}]: process error (code {rc}), retrying once from epoch 0 ...\n' + tail[-1500:], flush=True)
                rc, tail = train(r)
                if rc != 0 and not os.path.exists(rdir(*r) + '/summary.json'):
                    print('   second failure - stopping this session so that the error can be checked:\n' + tail, flush=True)
                    raise SystemExit(1)
        done = sum(os.path.exists(rdir(*q) + '/summary.json') for q in ALL_RUNS())
        if os.path.exists(rdir(*r) + '/summary.json'):
            S = json.load(open(rdir(*r) + '/summary.json'))
            print(time.strftime('%H:%M UTC', time.gmtime()), f"| {m} fold {f} seed {s}: Dice {S['dice_per_image']:.4f}, "
                  f"{S['epochs_run']} ep (best {S['best_epoch']}), {(time.time() - t0) / 60:.1f} min | finished {done} / {len(ALL_RUNS())}", flush=True)
        else:
            print(time.strftime('%H:%M UTC', time.gmtime()), f'| {m} fold {f} seed {s}: FAILED (recorded)', flush=True)
def ALL_RUNS(): return RUNS + (STEP2 or [])
phase(RUNS)
# E3 step 2: waits until every step-1 run is finished (the other session may still be training one)
while not all(finished(r) for r in STEP1):
    print(time.strftime('%H:%M UTC', time.gmtime()), '| waiting for the last E3 step-1 run(s) of the other session ...', flush=True); time.sleep(300)
SEL = select()
selfile = f'{OUT}/E3_selection.json'
if not os.path.exists(selfile): json.dump(SEL, open(selfile, 'w'), indent=1)
assert json.load(open(selfile)) == json.loads(json.dumps(SEL)), 'selection differs from the stored one'
print('E3 selection:', {b: v['selected'] for b, v in SEL.items()})
STEP2 = [(f"{b}_fast{v['selected']}", f, s, 'train') for b, v in SEL.items() if v['selected'] in (10, 30) for s in (1, 2) for f in range(5)]
print('E3 step 2 runs:', len(STEP2))
phase(STEP2)
left = [r for r in ALL_RUNS() if not finished(r)]
print(f'session {SESSION} has nothing left to start; {len(left)} run(s) still running in the other session')

In [ ]:
# 4 · Overview of the stage-4 runs (with block weight change)
import pandas as pd
rows = [dict(json.load(open(rdir(*r) + '/summary.json')), schedule=r[3]) for r in ALL_RUNS() if os.path.exists(rdir(*r) + '/summary.json')]
for r, row in zip([r for r in ALL_RUNS() if os.path.exists(rdir(*r) + '/summary.json')], rows):
    wc = rdir(*r) + '/weight_change.json'
    if os.path.exists(wc):
        W = json.load(open(wc))
        for ln in ('quantum_circuit', 'classical_replacement', 'trig_replacement'):
            if ln in W: row.update(slot_max_change=W[ln]['max_abs_change'], slot_rel_change=W[ln]['rel_change'])
ov = pd.DataFrame(rows); ov.to_csv(f'{OUT}/overview_stage4_{SESSION}.csv', index=False)
print(len(ov), 'of', len(ALL_RUNS()), 'finished;', sum(os.path.exists(rdir(*r) + '/failed.json') for r in ALL_RUNS()), 'failed')
if len(ov): print(ov.groupby(['model'])[[c for c in ('dice_per_image', 'epochs_run', 'best_epoch', 'slot_max_change', 'slot_rel_change') if c in ov]].mean().round(4).to_string())